# Index spread moves

Pick an index (`HY47`, `HY42`, `XO44`, `IG45`, `MA40`, ...) and two dates. Output is one row per index member with the starting spread, the ending spread and the change, sorted by the largest widening first.

- Reads the local store `cache/spreads.parquet` (no Databricks query). To add newer dates, run `cs.update_store()` (needs a running Bloomberg Terminal and a Databricks login) in the refresh cell below.
- Spread is `conv_spread` in bp at the index maturity. Dates snap to the last available date on or before the request, and an index has no data before its launch date.
- Names with no data on a date stay in the table with blank spreads and a `status` saying why: `missing` (no Databricks row) or `conv=0` (Markit publishes 0 when no spread solves, e.g. very wide names). A `spread_override` from `overrides.csv` replaces the value.

In [ ]:
import pandas as pd
import cdx_spreads as cs

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)


def show(df, numeric=()):
    """Render a table with no wrapping; `numeric` columns always show exactly one decimal (blank when missing)."""
    return (df.style
            .format({c: '{:.1f}' for c in numeric}, na_rep='')
            .set_properties(**{'white-space': 'nowrap'})
            .set_table_styles([{'selector': 'th', 'props': 'white-space: nowrap; text-align: left;'}]))


print(', '.join(cs.load_index_list()['index_name']))

## Refresh the store (optional)

In [ ]:
# cs.update_store()          # all 40 indices, only fetches dates the store doesn't have yet
# cs.update_store(['HY47'])  # or just some indices

## Parameters

In [ ]:
INDEX = 'HY47'
DATE_X = '2026-09-29'   # start
DATE_Y = '2026-10-23'   # end
SORT_BY_ABS = False     # False: widest first (largest positive change on top). True: largest move in either direction first.

## Run

In [ ]:
res = cs.spread_changes(INDEX, DATE_X, DATE_Y)
a = res.attrs
n_ok = (res['status'] == 'ok').sum()
print(f"{INDEX}: conv_spread {a['date_x']:%Y-%m-%d} -> {a['date_y']:%Y-%m-%d} | index maturity {a['maturity']:%Y-%m-%d} | {len(res)} names, {n_ok} with both spreads, {len(res) - n_ok} flagged")
print(f"median change {res['change'].median():.1f} bp | median pct change {res['pct_change'].median():.1f}%")

if SORT_BY_ABS:
    res = res.sort_values('change', key=abs, ascending=False, na_position='last').reset_index(drop=True)

view = res[['member_name', 'Market_CDS_Ticker', f"conv_spread_x", f"conv_spread_y", 'change', 'pct_change', 'status']].rename(columns={
    'conv_spread_x': f"spread {a['date_x']:%Y-%m-%d}", 'conv_spread_y': f"spread {a['date_y']:%Y-%m-%d}",
    'change': 'change (bp)', 'pct_change': 'change (%)'})
show(view, numeric=view.columns[2:6])

## What we don't have
Names flagged above: `missing` means Databricks has no row at this index's maturity on that date; `conv=0` means Markit published no conventional spread.

In [ ]:
flag = res[res['status'] != 'ok'][['member_name', 'Market_CDS_Ticker', 'x_state', 'y_state', 'status']]
print(res['status'].value_counts().to_string())
show(flag)